In [54]:
import pandas as pd
path='freedom_tele_ust.csv'
df=pd.read_csv(path)
df['город']='Усть-Каменогорск'
df.to_csv(path, index=False)

In [ ]:
import pandas as pd
path='freedom_bank_shym.csv'
df=pd.read_csv(path)
df['отделение']='Улица Казыбек би'
df.to_csv(path, index=False)

In [487]:
import spacy
import pandas as pd
import pandas as pd
df=pd.read_csv("freedom_insurance.csv", encoding="utf-8-sig")
nlp = spacy.load("ru_core_news_sm")  # если отзывы на русском

def extract_persons(text):
    if not isinstance(text, str):
        return []
    
    doc = nlp(text)
    
    return [
        ent.text.strip()
        
        for ent in doc.ents
        if ent.label_ == "PER" or ent.label_ == "PERSON"
    ]

df["люди"] = df["текст"].apply(extract_persons)


KeyboardInterrupt: 

In [475]:
df["review_id"] = df.index

In [476]:
import re
import numpy as np

def clean_people(x):
    if not isinstance(x, str):
        return np.nan   # пустое сохраняем как NaN

    x = x.lower().replace('\xa0', ' ')
    x = re.sub(r'[^\w\s]', '', x)

    tokens = x.split()

    # убираем мусорные токены (роли и т.п.)
    bad = {
        "менеджер", "ментор", "преподаватель",
        "саппорт", "академия", "финанс", "брокер",
        'самый',"супервайзер","маман","оте"
    }

    tokens = [t for t in tokens if t not in bad and len(t) > 1]

    if len(tokens) == 0:
        return np.nan

    return " ".join(tokens)


In [477]:
df['people']=df['люди']

In [478]:

df=df.explode('people')

In [479]:

df["people"] = df["people"].apply(clean_people)


In [ ]:
import re

phrase_to_remove = {
    'берди шешип',
    'банке фрид',
    'алдық үй',
    'ала алмай',
    "булар маған", "билетімді тауып", "возьмите курсы", "дегенбі басам",
    'ван лав','фурманова','эйр астаны','алматы астана','алматы новосибирск',
    'да мүмкін','алматы атырау','бірақ соңғы','өткізе алмай','алматы медина',
    'раз вы','ба мен','откенмин ба','салеметсиз бе',
    'калай канша','алуга болады','алуга болама','али акшасын',
    'отырмыз айдан', 'құзырлы органға', 'ісінің өз',
    'де тез', 'берді көмек', 'айтамын ке рахматымды', 'бақытты болсын көмектесті'
    , 'кыздар молодцы', 'берди шешип', 'берді шешіп', 
    'алғыс шексіз', 'байқалды танытқаны', 'бағалаймын деп',
    'ескеріп мүмкіндіктерімді','артык минутта сөзсіз',
    'берді жасап', 'ашылды рақметалғыс тез', 'жұмыс көрсетіп',
    'берді дайындап', 'кассир шырын', 'денгейд жогары',
    'асель ге', 'марал қызмет', 'берді түсіндіріп'
    
}

word_to_remove = {
    'өте', 'көп', 'фрид',
    'рахмет', 'барин', 'алла', 'фридома', "көңілім","ещё",
    "трэвел","тай", "банка", "бұл", "ура", 'вами', "благо","екен",
    "решил", "гагарина", 'сижу','блин','алматы','авиате',
    'билет','уму','брон','маган', 'бек', 'ух', 'банк',
    'кушты', 'маған', 'жаксы', 'әсірес', 'сотрудники',
    'керемет','фрид', 'тиімді', 'четко', 'атыра',
    'мен', 'ең', 'блогодорю', 'помог', 'аллах','конаев',
    'выражаю', 'менеджера', 'все', 'менеджеры', 'успехов',
    'приветливая', 'заблокировали', 'болса', 'бүгін',
    'карточка', 'ерекш', 'благодарю', 'берды', 'барлық',
    'улкен','көмектесті', 'ашып' , 'супервайзерг'
    'әркім', 'фридом'
}

import re
import numpy as np

phrase_to_remove = {p.lower() for p in phrase_to_remove}
word_to_remove = {w.lower() for w in word_to_remove}

def clean(name):
    if not isinstance(name, str):
        return np.nan

    name = name.lower()
    name = name.replace('\xa0', ' ')

    # --- 1. фразы (через regex безопаснее, чем replace)
    for ph in phrase_to_remove:
        name = re.sub(r'\b' + re.escape(ph) + r'\b', ' ', name)

    # --- 2. убрать пунктуацию
    name = re.sub(r'[^\w\s]', ' ', name)

    tokens = name.split()

    cleaned = []
    for t in tokens:
        if (
            t in word_to_remove or
            len(t) <= 1 or
            t.isdigit()
        ):
            continue
        cleaned.append(t)

    if len(cleaned) == 0:
        return np.nan


    return " ".join(cleaned)
df['people']=df['people'].apply(clean)

In [486]:
df['people'].value_counts()


people
айгерим                                  6
каско                                    2
фридом                                   2
владимир                                 2
назым                                    2
айгуль                                   2
страховой                                2
спасибо                                  1
владимир мешко                           1
русика                                   1
звонишь                                  1
дилер                                    1
данияр                                   1
царапина                                 1
алиной                                   1
буду                                     1
колл центр                               1
иншуранс                                 1
ели                                      1
шарашкина                                1
диана                                    1
давыдов константин                       1
выплата                                  1
диск

In [482]:
import re

def normalize_token(t):
    t = t.lower().strip()
    t = re.sub(r'[^\w\s]', '', t)

    # лёгкое снятие типичных окончаний (очень важно для RU/KZ данных)
    for suf in ["ом", "ым", "ему", "у", 
                'ға', 'тың', 'ның', 'га', 'ге']:
        if len(t) > 5 and t.endswith(suf):
            t = t[:-len(suf)]
            break


    return t
def normalize_person(name):
    if not isinstance(name, str):
        return name

    name = name.lower()
    name = name.replace('\xa0', ' ')
    name = re.sub(r'[^\w\s]', '', name)

    tokens = [normalize_token(t) for t in name.split()]
    
    # убираем пустые
    tokens = [t for t in tokens if len(t) > 1]

    if len(tokens) == 0:
        
        return None
    # сортировка убирает проблему "евгений каримов" vs "каримов евгений"
    tokens = sorted(tokens)

    return " ".join(tokens)
df["people"] = df["people"].apply(normalize_person)

In [457]:
def normalize_name(name):
    mapping = {
        'гульнара':'бухатановa гульнар',
        'бухатановой гульнар':'бухатановa гульнар',
        'бошаева':'айгуль бошаева',
        'сабин':'сабина',
        'руслана':'актанов руслан',
        'руслан':'актанов руслан',
        'еркежан төлеген':'базарбайқызы еркежан',
        'дане':'дана',
        'улбик':'ұлбике',
        'ұлбик': 'ұлбике'
    }
    
    return mapping.get(name, name)
import numpy as np


def safe_normalize(x):
    if isinstance(x, str):
        return normalize_name(x)

    return np.nan
df["people"] = df["people"].apply(safe_normalize)

In [459]:
df.drop(columns=['люди'], inplace=True)

In [460]:
df_reviews = df.drop_duplicates("review_id")


In [461]:
df_reviews.rename(columns={'people': 'люди'}, inplace=True)


C:\Users\Subbota\AppData\Local\Temp\ipykernel_3708\4174685759.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reviews.rename(columns={'people': 'люди'}, inplace=True)


In [4]:
df_reviews=pd.read_csv('freedom_bank.csv')

In [5]:
df_reviews['люди'].value_counts()

люди
жунусбаева асем      728
жайлаубаев дастан    168
алтаева жазира       126
дуйсенова айгерим     78
калибекова айнура     72
                    ... 
кутылу ушин            1
цел                    1
потом                  1
тугулбаева             1
сколько                1
Name: count, Length: 713, dtype: int64

In [ ]:
df_reviews.drop(columns=['review'
'_id'], inplace=True)

C:\Users\Subbota\AppData\Local\Temp\ipykernel_3708\4240686405.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reviews.drop(columns=['review'


In [6]:
top15 = df_reviews['люди'].value_counts().head(15
                                               ).index
df_reviews['люди'] = df_reviews['люди'].where(df_reviews['люди'].isin(top15))

In [7]:
df_reviews.columns

Index(['дата', 'время', 'рейтинг', 'текст', 'автор', 'likes_count',
       'official_answer', 'отделение', 'люди'],
      dtype='object')

In [8]:
df_reviews.to_csv('freedom_bank.csv', index=False)

In [13]:
import pandas as pd
path='freedom_ticketon.csv'

df=pd.read_csv(path)
df['город']='Алматы'

df.to_csv(path, index=False)